In [1]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune
import torch.nn.functional as F

## 1. CONFIGURACIÓN

In [5]:
DATASET = "FD001"

RANDOM_STATE = 42

# Cada cuántas épocas se mide la validación durante la búsqueda.
SEARCH_VALIDATION_INTERVAL = 25

# Entrenamiento durante la búsqueda.
SEARCH_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# El early stopping se activa después de este mínimo de épocas para no detener
# prematuramente redes pequeñas que pueden aprender con mayor lentitud.
FINAL_MIN_EPOCHS_BEFORE_STOP = 500

# Dispositivo. En GPU, las matrices de este MLP son pequeñas, pero el
# entrenamiento de múltiples arquitecturas puede acelerarse.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Archivos de salida.
CACHE_VERSION = 6
OBJECTIVE_NAME = "rmse"
if OBJECTIVE_NAME not in {"rmse", "nasa", "both"}:
    raise ValueError("OBJECTIVE_NAME debe ser 'rmse', 'nasa' o 'both'.")
# En modo both, RMSE elige el checkpoint; MOPSO optimiza RMSE, NASA y parámetros.
CACHE_DIR = Path("mopso_cache")
RESULTS_DIR = Path("mopso_results")
MODELS_DIR = Path("mopso_models")

# Configuraciones para la poda
PRUNING_FINE_TUNE_EPOCHS = 1_000
PRUNING_VALIDATION_INTERVAL = 25
PRUNING_EARLY_STOPPING_PATIENCE = 10
PRUNING_MIN_EPOCHS_BEFORE_STOP = 100
PRUNING_ALPHAS = [1, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60, 0.55, 0.50]

sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size": 12, "figure.dpi": 180})

## 2. CARGA DE LOS MODELOS

In [6]:
from collections import OrderedDict


def build_candidate_model(candidate: Dict[str, Any]) -> nn.Sequential:
    """Reconstruye un candidato JSON como una red densa torch.nn."""
    representative = candidate.get("representative_model")
    if not isinstance(representative, dict):
        raise ValueError(
            f"El candidato {candidate.get('candidate_id')} "
            "no tiene un representative_model válido."
        )

    layer_configs = representative.get("layers")
    if not isinstance(layer_configs, list) or not layer_configs:
        raise ValueError(
            f"El candidato {candidate.get('candidate_id')} no tiene capas."
        )

    activation_factories = {
        "linear": nn.Identity,
        "identity": nn.Identity,
        "relu": nn.ReLU,
        "elu": nn.ELU,
        "silu": nn.SiLU,
        "tanh": nn.Tanh,
        "sigmoid": nn.Sigmoid,
    }
    modules = OrderedDict()
    previous_size = None

    for layer_index, layer_config in enumerate(layer_configs):
        if not isinstance(layer_config, dict):
            raise ValueError(f"La capa {layer_index} debe ser un objeto JSON.")
        if layer_config.get("type", "dense") != "dense":
            raise ValueError(
                f"Tipo de capa no soportado: {layer_config.get('type')!r}."
            )
        if layer_config.get(
            "weights_layout", "[output_neuron][input_neuron]"
        ) != "[output_neuron][input_neuron]":
            raise ValueError(
                f"Formato de pesos no soportado en la capa {layer_index}."
            )

        weights = torch.as_tensor(
            layer_config.get("weights"),
            dtype=torch.float32,
        )
        bias = torch.as_tensor(layer_config.get("bias"), dtype=torch.float32)
        if weights.ndim != 2:
            raise ValueError(f"Los pesos de la capa {layer_index} deben ser 2D.")
        output_size, input_size = weights.shape
        if (
            input_size != int(layer_config.get("input_size", -1))
            or output_size != int(layer_config.get("output_size", -1))
        ):
            raise ValueError(
                f"Las dimensiones declaradas no coinciden con los pesos "
                f"de la capa {layer_index}."
            )
        if previous_size is not None and input_size != previous_size:
            raise ValueError(
                f"La capa {layer_index} espera {input_size} entradas, "
                f"pero la capa anterior produce {previous_size}."
            )
        if bias.shape != (output_size,):
            raise ValueError(
                f"El bias de la capa {layer_index} debe tener "
                f"dimensión ({output_size},)."
            )
        if not torch.isfinite(weights).all() or not torch.isfinite(bias).all():
            raise ValueError(f"La capa {layer_index} contiene valores no finitos.")

        linear = nn.Linear(input_size, output_size, bias=True)
        with torch.no_grad():
            linear.weight.copy_(weights)
            linear.bias.copy_(bias)
        modules[f"linear_{layer_index}"] = linear

        activation_name = str(layer_config.get("activation", "linear")).lower()
        activation_factory = activation_factories.get(activation_name)
        if activation_factory is None:
            raise ValueError(
                f"Activación no soportada en la capa {layer_index}: "
                f"{activation_name}."
            )
        modules[f"activation_{layer_index}"] = activation_factory()

        dropout_probability = float(layer_config.get("dropout_after", 0.0))
        if not 0.0 <= dropout_probability < 1.0:
            raise ValueError(
                f"Dropout inválido en la capa {layer_index}: "
                f"{dropout_probability}."
            )
        if dropout_probability > 0.0:
            modules[f"dropout_{layer_index}"] = nn.Dropout(
                p=dropout_probability
            )

        previous_size = output_size

    return nn.Sequential(modules).to(DEVICE)


candidates_path = RESULTS_DIR / (
    f"{DATASET}_mopso_final_candidates_v{CACHE_VERSION}_"
    f"{OBJECTIVE_NAME}.json"
)
if not candidates_path.is_file():
    raise FileNotFoundError(f"No se encontró el JSON de candidatos: {candidates_path}")

with candidates_path.open("r", encoding="utf-8") as json_file:
    models_config = json.load(json_file)

if models_config.get("format") != "agnostic-dense-mlp":
    raise ValueError("El archivo no contiene un JSON agnóstico de MLP compatible.")
if int(models_config.get("format_version", -1)) != 1:
    raise ValueError(
        f"Versión de JSON no soportada: {models_config.get('format_version')}."
    )
if models_config.get("dataset") != DATASET:
    raise ValueError(
        f"El JSON pertenece a {models_config.get('dataset')}, "
        f"no a {DATASET}."
    )

candidate_configs = models_config.get("candidates")
if not isinstance(candidate_configs, list) or not candidate_configs:
    raise ValueError("El archivo JSON no contiene candidatos para reconstruir.")

candidate_models = []
for candidate_config in candidate_configs:
    model = build_candidate_model(candidate_config)
    model.eval()
    candidate_models.append(
        {
            "candidate_id": candidate_config["candidate_id"],
            "selected_winner": bool(candidate_config.get("selected_winner", False)),
            "representative_seed": int(
                candidate_config["representative_model"]["seed"]
            ),
            "architecture": candidate_config["architecture"],
            "parameter_count": int(candidate_config["parameter_count"]),
            "test_results": candidate_config.get("test_results", {}),
            "model": model,
        }
    )

print(f"Modelos reconstruidos: {len(candidate_models)}")
for candidate in candidate_models:
    print(
        f"{candidate['candidate_id']} | seed={candidate['representative_seed']} | "
        f"winner={candidate['selected_winner']} | "
        f"architecture={candidate['architecture']['hidden_sizes']}"
    )

Modelos reconstruidos: 4
candidate_01 | seed=44 | winner=False | architecture=[24, 48, 64]
candidate_02 | seed=33 | winner=False | architecture=[4]
candidate_03 | seed=55 | winner=True | architecture=[16, 4, 48]
candidate_04 | seed=44 | winner=False | architecture=[4, 4, 8]


## 3. PODA

$$
\min z = RMSE_{\mathrm{val}}
$$

sujeto a

$$
A_{\mathrm{activo}}
\leq
\left\lfloor
\alpha A_0
\right\rfloor
$$

con

$$
0 < \alpha \leq 1
$$

## Poda de arcos mediante una restricción de esparsidad

Una vez seleccionadas las arquitecturas candidatas de la MLP mediante MOPSO, se introduce una segunda etapa de optimización destinada a reducir el número de conexiones activas de la red.

En esta etapa, la topología a nivel de capas permanece fija. La poda actúa sobre las conexiones individuales de las capas `Linear`, eliminando aquellos pesos que se consideran menos relevantes.

### Definición de arco

Para una MLP con dimensión de entrada $n_0$, capas ocultas de tamaños

$$
(n_1,n_2,\ldots,n_L)
$$

y una salida escalar $n_{L+1}=1$, el número de arcos de la red densa original es

$$
A_0 =
\sum_{\ell=1}^{L+1}
n_{\ell-1}n_{\ell}
$$

Cada elemento de una matriz de pesos $W^{(\ell)}$ representa un arco dirigido entre dos neuronas.

Los términos de sesgo (`bias`) no se consideran arcos. Por tanto, esta medida es diferente de `n_parameters`, utilizada anteriormente en el MOPSO, ya que `n_parameters` incluye tanto pesos como sesgos.

### Variables de poda

Para cada peso $w_{ij}^{(\ell)}$ se define una variable binaria

$$
m_{ij}^{(\ell)} \in {0,1},
$$

donde

$$
m_{ij}^{(\ell)} =
\begin{cases}
1, & \text{si el arco se conserva},\
0, & \text{si el arco se elimina}.
\end{cases}
$$

La matriz efectiva de pesos de cada capa se define entonces como

$$
\widetilde{W}^{(\ell)}
=

W^{(\ell)} \odot M^{(\ell)},
$$

donde $\odot$ representa el producto elemento a elemento.

### Modelo de optimización

Sea $\alpha$ la fracción máxima de arcos de la red original que pueden permanecer activos, con

$$
0 < \alpha \leq 1.
$$

Para un valor de $\alpha$ fijado previamente, el problema de poda puede escribirse como

$$
\min_{W,M} \quad
z =
RMSE_{\mathrm{val}}
\left(
y,
f(X,W\odot M,b)
\right)
$$

sujeto a

$$
\sum_{\ell=1}^{L+1}
\sum_{i=1}^{n_{\ell-1}}
\sum_{j=1}^{n_{\ell}}
m_{ij}^{(\ell)}
\leq
\left\lfloor
\alpha A_0
\right\rfloor,
$$

$$
m_{ij}^{(\ell)}\in{0,1}.
$$

De forma explícita, el objetivo utilizado es

$$
RMSE_{\mathrm{val}}
=

\sqrt{
\frac{1}{N_{\mathrm{val}}}
\sum_{k=1}^{N_{\mathrm{val}}}
\left(
y_k-\hat y_k
\right)^2
}.
$$

Para mantener la misma definición empleada durante la búsqueda MOPSO, las predicciones de validación pueden limitarse al intervalo de RUL:

$$
\hat y_k =
\operatorname{clip}
\left(
f(x_k),0,RUL_{\max}
\right),
\qquad
RUL_{\max}=125.
$$

### Interpretación de $\alpha$

En esta formulación, $\alpha$ representa la **fracción máxima de arcos conservados**, no el número absoluto de arcos.

Por ejemplo,

$$
\alpha=0.70
$$

significa que como máximo puede permanecer activo el $70%$ de los arcos originales, por lo que al menos el $30%$ debe ser eliminado.

Si

$$
A_0=5000,
$$

entonces

$$
A_{\mathrm{activo}}
\leq
\lfloor0.70(5000)\rfloor
=

3500.

$$

La esparsidad obtenida puede expresarse como

$$
S
=

1-
\frac{A_{\mathrm{activo}}}{A_0}.
$$

Por tanto, idealmente,

$$
S \geq 1-\alpha.
$$

### Estrategia práctica de solución

La optimización conjunta de todos los pesos y todas las variables binarias $m_{ij}^{(\ell)}$ constituye un problema combinatorio no convexo de gran dimensión.

Por esta razón, se utiliza una aproximación en dos etapas:

1. Entrenar y seleccionar primero la arquitectura densa mediante el MOPSO existente.
2. Ordenar globalmente los pesos de las capas `Linear` de acuerdo con su magnitud absoluta.
3. Conservar únicamente los

   $$
   K=\lfloor\alpha A_0\rfloor
   $$

   pesos de mayor magnitud.
4. Asignar máscara cero al resto de los pesos.
5. Mantener las máscaras fijas y realizar un proceso corto de *fine-tuning*.
6. Seleccionar el checkpoint con menor RMSE de validación.
7. Evaluar el conjunto de test solamente después de haber fijado tanto la arquitectura como el nivel de poda.

Esta estrategia corresponde a **global unstructured magnitude pruning**, porque la comparación de magnitudes se realiza globalmente entre todas las capas y se eliminan conexiones individuales.

El código de este cuaderno genera una copia podada de cada candidato para cada valor de `PRUNING_ALPHAS`. Estas variantes conservan las activaciones y los biases originales; el fine-tuning y la evaluación de validación/test son etapas posteriores y no se ejecutan en esta celda.

### Relación con MOPSO

La búsqueda MOPSO actual determina

$$
(\text{profundidad},
\text{anchuras},
\text{activación},
\text{dropout})
$$

utilizando como objetivos el RMSE y el número total de parámetros.

La poda se ejecuta posteriormente sobre la solución seleccionada:

$$
\text{MOPSO}
\longrightarrow
\text{MLP densa seleccionada}
\longrightarrow
\text{poda}
\longrightarrow
\text{fine-tuning}
\longrightarrow
\text{modelo final}.
$$

Se evita introducir una variable binaria por cada arco dentro de la codificación del PSO, ya que esto aumentaría la dimensión de búsqueda desde unas pocas variables arquitectónicas hasta potencialmente miles de variables binarias y además interferiría con el mecanismo actual de caché y *warm start*.

### Selección de $\alpha$

Si $\alpha$ se considera una variable de decisión y el único objetivo es minimizar el RMSE, la solución tendería naturalmente hacia

$$
\alpha \rightarrow 1,
$$

es decir, hacia una red sin poda.

Por ello, $\alpha$ debe:

* fijarse como un presupuesto de complejidad; o
* evaluarse sobre un conjunto discreto de valores utilizando únicamente validación; o
* incorporarse como un segundo objetivo.

Una posible extensión multiobjetivo sería

$$
\min
\left(
RMSE_{\mathrm{val}},
\frac{A_{\mathrm{activo}}}{A_0}
\right).
$$

Esto permitiría obtener un nuevo frente de Pareto entre precisión predictiva y esparsidad.

La opción seleccionada con respecto a $\alpha$, al menos por ahora, es la de evaluarse sobre un conjunto discreto de valores.

In [7]:
def prune_candidate_model(
    model: nn.Module,
    alpha: float,
) -> Tuple[nn.Module, Dict[str, int | float]]:
    """Prune globally by weight magnitude without modifying the source model."""
    alpha = float(alpha)
    if not math.isfinite(alpha) or not 0.0 < alpha <= 1.0:
        raise ValueError("Cada alpha debe cumplir 0 < alpha <= 1.")

    pruned_model = copy.deepcopy(model).to(DEVICE)
    linear_layers = [
        layer for layer in pruned_model.modules()
        if isinstance(layer, nn.Linear)
    ]
    if not linear_layers:
        raise ValueError("El modelo candidato no contiene capas nn.Linear.")

    parameters_to_prune = [(layer, "weight") for layer in linear_layers]
    total_arcs = sum(int(layer.weight.numel()) for layer in linear_layers)
    target_active_arcs = int(math.floor(alpha * total_arcs))
    target_pruned_arcs = total_arcs - target_active_arcs

    if target_pruned_arcs:
        prune.global_unstructured(
            parameters_to_prune,
            pruning_method=prune.L1Unstructured,
            amount=target_pruned_arcs,
        )
        for layer in linear_layers:
            prune.remove(layer, "weight")

    active_arcs = sum(
        int(torch.count_nonzero(layer.weight.detach()).item())
        for layer in linear_layers
    )
    pruning_summary = {
        "alpha": alpha,
        "total_arcs": total_arcs,
        "target_active_arcs": target_active_arcs,
        "active_arcs": active_arcs,
        "pruned_arcs": total_arcs - active_arcs,
        "sparsity": 1.0 - active_arcs / total_arcs,
    }
    pruned_model.eval()
    return pruned_model, pruning_summary


if not candidate_models:
    raise ValueError("No hay modelos candidatos cargados para podar.")
if not PRUNING_ALPHAS:
    raise ValueError("PRUNING_ALPHAS no puede estar vacío.")

pruned_candidate_models = []
pruning_summaries = []
for candidate in candidate_models:
    for alpha in PRUNING_ALPHAS:
        pruned_model, pruning_summary = prune_candidate_model(
            model=candidate["model"],
            alpha=alpha,
        )
        record = {
            "candidate_id": candidate["candidate_id"],
            "selected_winner": candidate["selected_winner"],
            "representative_seed": candidate["representative_seed"],
            "architecture": candidate["architecture"],
            **pruning_summary,
            "model": pruned_model,
        }
        pruned_candidate_models.append(record)
        pruning_summaries.append(
            {key: value for key, value in record.items() if key != "model"}
        )

pruning_results = pd.DataFrame(pruning_summaries)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pruning_results.to_csv(
    RESULTS_DIR / f"{DATASET}_candidate_pruning_summary.csv",
    index=False,
)
print(
    f"Variantes podadas: {len(pruned_candidate_models)} "
    f"({len(candidate_models)} candidatos × {len(PRUNING_ALPHAS)} alphas)."
)
print(pruning_results)

Variantes podadas: 44 (4 candidatos × 11 alphas).
    candidate_id  selected_winner  representative_seed  \
0   candidate_01            False                   44   
1   candidate_01            False                   44   
2   candidate_01            False                   44   
3   candidate_01            False                   44   
4   candidate_01            False                   44   
5   candidate_01            False                   44   
6   candidate_01            False                   44   
7   candidate_01            False                   44   
8   candidate_01            False                   44   
9   candidate_01            False                   44   
10  candidate_01            False                   44   
11  candidate_02            False                   33   
12  candidate_02            False                   33   
13  candidate_02            False                   33   
14  candidate_02            False                   33   
15  candidate_02      